# 🚀 Entraînement Baselines - DLinear, PatchTST, PatchMixer

Entraîne les 3 modèles et compare leurs performances.

In [ ]:
import sys
import os

# Se placer à la racine du projet pour les chemins relatifs
os.chdir('..')
sys.path.append('.')

import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from tqdm.notebook import tqdm

# Réutiliser le code existant
from exp.exp_main import Exp_Main
from tools.metrics import metric

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
print(f"Working directory: {os.getcwd()}")

## Configuration

In [ ]:
# Charger une config de base (on utilisera linearModel comme template)
with open('configs/linearModel.json', 'r') as f:
    base_config = json.load(f)

# Ajuster pour entraînement rapide (optionnel)
base_config['epochs'] = 30  # Réduire pour tests rapides
base_config['patience'] = 5

print(f"Configuration chargée:")
print(f"  - Epochs: {base_config['epochs']}")
print(f"  - Batch size: {base_config['batch_size']}")
print(f"  - Data: {base_config['data']}")

class Args:
    def __init__(self, config_dict):
        for key, value in config_dict.items():
            setattr(self, key, value)

## Entraînement des 3 modèles

In [ ]:
models = ['DLinear', 'PatchTST', 'PatchMixer']
results = {}

# Créer le dossier checkpoints s'il n'existe pas
os.makedirs('checkpoints', exist_ok=True)

for model_name in models:
    print(f"\n{'='*60}")
    print(f"Entraînement {model_name}")
    print('='*60)
    
    # Config
    config = base_config.copy()
    config['model'] = model_name
    config['tracking_gradcam'] = False  # Désactiver pour rapidité
    args = Args(config)
    
    # Entraîner avec code existant
    start = time.time()
    exp = Exp_Main(args, device, args.data)
    exp.training()
    
    # Tester
    preds, trues, preds_last, trues_last, _ = exp.testing()
    metrics_test = metric(trues, preds)
    metrics_val = exp.validating()
    
    elapsed = time.time() - start
    
    results[model_name] = {
        'test': metrics_test,
        'val': metrics_val,
        'time': elapsed
    }
    
    # Sauvegarder le modèle pour le notebook d'explicabilité
    checkpoint_path = f'checkpoints/{model_name}_baseline.pth'
    torch.save(exp.model_pred.state_dict(), checkpoint_path)
    print(f"✓ Modèle sauvegardé: {checkpoint_path}")
    
    print(f"\nTest  - MAE: {metrics_test[0]:.4f} | RMSE: {metrics_test[2]:.4f} | KGE: {metrics_test[7]:.4f}")
    print(f"Val   - MAE: {metrics_val[0]:.4f} | RMSE: {metrics_val[2]:.4f} | KGE: {metrics_val[7]:.4f}")
    print(f"Temps: {int(elapsed//60)}m{int(elapsed%60):02d}s")

## Comparaison

In [ ]:
# Tableau comparatif
df = pd.DataFrame([
    {
        'Modèle': name,
        'MAE': res['test'][0],
        'RMSE': res['test'][2],
        'KGE': res['test'][7],
        'Temps (s)': int(res['time'])
    }
    for name, res in results.items()
])

print("\n" + "="*60)
print("COMPARAISON DES MODÈLES")
print("="*60)
print(df.to_string(index=False))

In [ ]:
# Visualisation
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, (metric, metric_idx) in enumerate([('MAE', 0), ('RMSE', 2), ('KGE', 7)]):
    ax = axes[idx]
    values = [results[m]['test'][metric_idx] for m in models]
    bars = ax.bar(models, values, color=['#FF6B6B', '#4ECDC4', '#45B7D1'])
    ax.set_ylabel(metric)
    ax.set_title(f'Comparaison {metric}')
    ax.grid(True, alpha=0.3, axis='y')
    
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
                f'{val:.3f}', ha='center', va='bottom')

plt.tight_layout()

# Créer le dossier figs/ s'il n'existe pas
import os
os.makedirs('figs', exist_ok=True)
plt.savefig('figs/baselines_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## Sauvegarde

In [ ]:
import os
os.makedirs('results', exist_ok=True)
df.to_csv('results/baselines_comparison.csv', index=False)
print("✓ Résultats sauvegardés")